# Class 1 — Spark Batch Processing (EMR)

This notebook explains **how Apache Spark executes batch jobs**, from the programming model down to the physical execution engine. It is meant to be read top to bottom, then run cell by cell.

## 1. What problem Spark solves

A single machine cannot hold or process a dataset that is terabytes in size in reasonable time. Spark solves this by:

- **Partitioning** data across many machines (executors).
- Describing computation as a **logical plan** (a DAG of transformations) rather than executing eagerly.
- **Optimizing** that plan (Catalyst optimizer) before generating code (Tungsten/whole-stage codegen).
- **Scheduling** the optimized plan as stages of parallel tasks across the cluster.

## 2. Architecture

```text
               +------------------+
               |      Driver       |   <- builds logical plan, negotiates resources,
               |  (SparkContext)   |      schedules tasks, collects results
               +---------+--------+
                         |
         +---------------+----------------+
         |               |                |
 +-------v------+ +------v-------+ +------v-------+
 |  Executor 1   | |  Executor 2   | |  Executor N   |
 |  (JVM proc)   | |  (JVM proc)   | |  (JVM proc)   |
 |  tasks/cores  | |  tasks/cores  | |  tasks/cores  |
 |  partitions   | |  partitions   | |  partitions   |
 +---------------+ +---------------+ +---------------+
```

- The **driver** runs your code, builds the DAG, and asks the cluster manager (EMR: YARN, via Livy for this notebook's session) for executors.
- **Executors** run **tasks** -- one task per partition per stage. Executors hold data in memory/disk (cache) and report results/status back to the driver.
- A **job** is triggered by an **action** (e.g. `count()`, `collect()`, `write`). A job is split into **stages**, and stages are split by **shuffle boundaries**. Each stage is a set of **tasks** that can run without moving data between partitions.

## 3. Transformations vs actions (laziness)

- **Transformations** (`select`, `filter`, `withColumn`, `join`, `groupBy`) build the logical plan. Nothing executes yet.
- **Actions** (`count`, `collect`, `show`, `write.save`, `toTable`) trigger execution of the plan built so far.

This laziness lets Catalyst see the *entire* plan before deciding how to execute it -- e.g. it can push a filter below a join (predicate pushdown) even though you wrote the filter after the join in your code.

## 4. Narrow vs wide transformations (why shuffle matters)

- **Narrow**: each output partition depends on exactly one input partition (`select`, `filter`, `withColumn`, `union`, a join against a **broadcast** table). No data movement across the network.
- **Wide**: an output partition depends on *many* input partitions, which requires a **shuffle** -- data is repartitioned and moved across the network/disk (`groupBy`, `distinct`, `repartition`, a sort-merge join).

Shuffles are the single biggest cost lever in Spark: they involve disk I/O, network I/O, and serialization. Most performance tuning is really "reduce or reshape shuffles."

## 5. Catalyst, Tungsten, and Adaptive Query Execution (AQE)

- **Catalyst** is the logical/physical query optimizer: it rewrites your DataFrame/SQL plan (predicate pushdown, column pruning, constant folding, join reordering) before picking a physical strategy (broadcast-hash-join vs sort-merge-join, etc.).
- **Tungsten** is the execution engine: off-heap binary row format, whole-stage code generation.
- **AQE** (on by default in modern Spark) re-optimizes the plan *during* execution using actual runtime statistics: it can coalesce many small shuffle partitions into fewer larger ones, switch a sort-merge join to a broadcast join if a table turns out to be small, and split skewed partitions.

## 6. Partitioning

- **In-memory partitions** control parallelism (`spark.sql.shuffle.partitions`, `repartition`, `coalesce`). Too few partitions under-utilizes the cluster; too many creates task-scheduling overhead and tiny output files.
- **On-disk partitioning** (`partitionBy("event_date")` when writing) creates a directory-per-value layout so downstream readers can skip irrelevant data (partition pruning).

Below, we run these concepts against a small synthetic clickstream dataset.

Run the cell below first -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "class_emr"
base_path = "s3://<your-lakehouse-bucket>/data/class-emr"  # from `terraform output lakehouse_bucket_name`

def table(name):
    return f"`{schema}`.`{name}`"

def path(*parts):
    return base_path.rstrip("/") + "/" + "/".join(p.strip("/") for p in parts)

from pyspark.sql import functions as F

spark.sql(f"CREATE DATABASE IF NOT EXISTS `{schema}` LOCATION '{path('tables')}'")
spark.sql(f"USE `{schema}`")
print(f"schema={schema}, base_path={base_path}")

## Generate a small synthetic dataset

We generate deterministic data in-notebook so this class runs with no external dependency.

In [ ]:
import random
from datetime import datetime, timedelta, timezone

random.seed(7)

products = [
    (f"p{i:04d}", random.choice(["electronics", "grocery", "home", "apparel", "beauty"]),
     random.choice(["acme", "northstar", "evergreen", "summit", "nova"]), round(random.uniform(3, 500), 2))
    for i in range(1, 51)
]
customers = [
    (f"u{i:05d}", random.choice(["new", "active", "loyal", "at_risk"]), random.choice(["west", "central", "south", "east"]))
    for i in range(1, 201)
]
events = []
start = datetime.now(timezone.utc) - timedelta(hours=2)
for i in range(200_000):
    event_type = random.choice(["view", "add_to_cart", "purchase", "search", "checkout"])
    qty = random.randint(1, 4) if event_type in ("purchase", "checkout") else None
    price = round(random.uniform(5, 250), 2) if qty else None
    events.append((
        f"evt-{i:08d}",
        start + timedelta(seconds=random.randint(0, 7200)),
        random.choice(customers)[0],
        random.choice(products)[0],
        event_type, qty, price,
    ))

products_df = spark.createDataFrame(products, "product_id string, category string, brand string, price double")
customers_df = spark.createDataFrame(customers, "user_id string, segment string, region string")
events_df = spark.createDataFrame(
    events, "event_id string, event_ts timestamp, user_id string, product_id string, event_type string, quantity int, price double"
)

print("Event rows:", events_df.count())
num_partitions = events_df.select(F.spark_partition_id().alias("p")).distinct().count()
print("Default partitions of events_df:", num_partitions)

## Laziness in action

The next cell defines transformations only. Nothing runs yet -- Spark just extends the logical plan.

In [ ]:
normalized = (
    events_df
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_hour", F.date_trunc("hour", F.col("event_ts")))
    .withColumn("quantity", F.coalesce(F.col("quantity"), F.lit(0)))
    .withColumn("price", F.coalesce(F.col("price"), F.lit(0.0)))
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("price"), 2))
)
print(type(normalized))  # still just a DataFrame wrapping a logical plan -- no job has run

Calling `.count()` is an **action**: it triggers a job. Open the Spark UI (via the EMR console's "Application user interfaces" link, or Livy's session UI) and look at the Jobs/Stages page while this cell runs -- you'll see one job with one stage (narrow transformations only, no shuffle).

In [ ]:
print("Row count (triggers a job):", normalized.count())

## Narrow vs wide: broadcast join (narrow) vs groupBy (wide)

`products_df` is tiny (50 rows), so Spark (via AQE, or explicitly via `F.broadcast`) sends the *whole* table to every executor instead of shuffling the large `events` table. This turns what would otherwise be a wide join into a narrow one -- no shuffle for the join itself.

The subsequent `groupBy` **is** wide: rows with the same grouping key can live on any executor, so Spark must shuffle data so each key's rows land on one executor before aggregating.

In [ ]:
enriched = normalized.join(F.broadcast(products_df), "product_id", "left")

revenue_by_category_hour = (
    enriched
    .filter(F.col("event_type").isin("purchase", "checkout"))
    .groupBy("event_hour", "category")
    .agg(F.count("*").alias("orders"), F.round(F.sum("gross_amount"), 2).alias("revenue"))
)

revenue_by_category_hour.explain("formatted")

Read the plan above bottom-up. You should see:

- A `BroadcastHashJoin` (no shuffle for the join -- one side was small enough to broadcast).
- A `HashAggregate` -> `Exchange` (shuffle) -> `HashAggregate` pair around the `groupBy`. Spark pre-aggregates partially on each executor (the first `HashAggregate`) *before* shuffling, then finishes the aggregation after the shuffle (the second `HashAggregate`) -- this is a map-side combine, the same trick MapReduce combiners use, and it drastically cuts shuffle volume.

In [ ]:
revenue_by_category_hour.orderBy(F.desc("revenue")).limit(20).toPandas()

## Partition count and file sizing

`spark.sql.shuffle.partitions` controls how many partitions a shuffle produces. The default (200) is usually wrong for small clusters/datasets -- it creates many tiny tasks and tiny output files. AQE's `coalescePartitions` feature fixes this automatically at runtime in modern Spark, but it's important to understand the knob.

In [ ]:
print("Current shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
spark.conf.set("spark.sql.shuffle.partitions", "8")
print("Set to 8 for this small demo dataset. In production, size this to (cluster core count) x 2-3, "
      "or leave AQE to coalesce automatically.")

## Writing partitioned output (on-disk partitioning)

`partitionBy("event_date")` writes one directory per date. A downstream reader that filters on `event_date` can skip entire directories (partition pruning) without reading their data at all.

In [ ]:
(
    normalized.join(F.broadcast(products_df), "product_id", "left")
    .join(customers_df, "user_id", "left")
    .write.mode("overwrite")
    .partitionBy("event_date")
    .format("delta")
    .saveAsTable(table("class_batch_bronze_events"))
)

spark.sql(f"DESCRIBE DETAIL {table('class_batch_bronze_events')}").toPandas()

## Common batch performance problems (talking points)

1. **Too many small files** -- usually caused by over-partitioning on write or many small streaming micro-batches. Fix with `OPTIMIZE` (Delta) or by reducing output partition count.
2. **Data skew** -- one join/grouping key has far more rows than others, so one task takes far longer than the rest. AQE's skew join optimization splits the largest partitions automatically; salting keys is the manual fix when AQE isn't enough.
3. **Wrong join strategy** -- broadcasting a table that's actually large causes executor OOMs; failing to broadcast a genuinely small table causes an unnecessary shuffle. Check `explain()` to confirm.
4. **Repeated scans** -- reading the same source multiple times instead of caching an intermediate result that's reused. Cache deliberately, and `unpersist()` when done.
5. **Missing predicate/column pushdown** -- reading Parquet/Delta with a `filter` before a `select` lets Spark skip files/row-groups and unused columns entirely at the source. `explain()` shows `PushedFilters`.

## What's next

Batch processing assumes the data is already fully available. `02_spark_structured_streaming.ipynb` covers what changes when data arrives continuously and you cannot wait for "all of it."